In [0]:
SHOW TABLES IN workspace.default;

In [0]:
SELECT *
FROM tp_entries_raw
LIMIT 10;

In [0]:
SELECT COUNT(*) AS total_entries
FROM tp_entries_raw;

total_entries
133


In [0]:
SELECT *
FROM tp_attractions;

attraction_id,attraction_name,capacity_per_hour,category,features,is_active,location,min_height_cm,opened_year
A01,Sky Spinner,140,Thrill,"List(Outdoor, High Speed)",true,"List(N1, North)",120,2018
A02,Coral Carousel,200,Family,"List(Indoor, Family Friendly)",true,"List(E2, East)",0,2015
A03,Jungle Rapids,110,Water,"List(Water Ride, Outdoor)",true,"List(S1, South)",110,2020
A04,Moonlight Train,180,Family,"List(Scenic, Family Friendly)",true,"List(W3, West)",0,2012
A05,Meteor Drop,100,Thrill,"List(High Speed, Outdoor)",true,"List(N2, North)",135,2022
A06,Pirate Voyage,130,Water,"List(Water Ride, Themed)",true,"List(E1, East)",100,2019
A07,Crystal Maze,160,Adventure,"List(Indoor, Puzzle)",true,"List(S3, South)",90,2021
A08,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A09,Galaxy Theater,260,Show,"List(Indoor, Live Show)",true,"List(N3, North)",0,2017
A10,Mini Explorers,190,Family,"List(Kids, Indoor)",true,"List(E3, East)",0,2016


In [0]:
SELECT *
FROM tp_maintenance;

maintenance_id,attraction_id,maintenance_date,issue_type,downtime_minutes,maintenance_status
M001,A05,2026-10-15,Cleaning,90,Scheduled
M002,A08,2026-10-05,Mechanical,60,Completed
M003,A02,2026-10-12,Mechanical,0,Scheduled
M004,A09,2026-10-18,Electrical,0,Open
M005,A05,2026-10-12,Electrical,0,Scheduled
M006,A04,2026-10-07,Water System,45,Completed
M007,A10,2026-10-20,Mechanical,120,Completed
M008,A04,2026-10-04,Water System,90,Open
M009,A08,2026-10-19,Safety Check,30,Scheduled
M010,A02,2026-10-15,Safety Check,0,Open


jjj


In [0]:
CREATE OR REPLACE TABLE park_entries_bronze
USING DELTA
AS
SELECT *
FROM tp_entries_raw;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) AS total_records
FROM park_entries_bronze;

total_records
133


In [0]:
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN entry_id IS NULL THEN 1 ELSE 0 END) AS missing_entry_ids,
    SUM(CASE WHEN attraction_id IS NULL THEN 1 ELSE 0 END) AS missing_attraction_ids,
    SUM(CASE WHEN booking_channel IS NULL THEN 1 ELSE 0 END) AS missing_channels,
    SUM(CASE WHEN ticket_price IS NULL OR ticket_price <= 0 THEN 1 ELSE 0 END) AS invalid_prices,
    SUM(CASE WHEN wait_minutes IS NULL OR wait_minutes < 0 THEN 1 ELSE 0 END) AS invalid_wait_times
FROM park_entries_bronze;

total_records,missing_entry_ids,missing_attraction_ids,missing_channels,invalid_prices,invalid_wait_times
133,0,0,3,3,2


In [0]:
CREATE OR REPLACE TABLE park_entries_silver
USING DELTA
AS
SELECT
    entry_id,
    visitor_id,
    attraction_id,
    visit_date,
    entry_time,
    wait_minutes,
    ride_minutes,
    ticket_type,
    ticket_price,
    status,
    COALESCE(booking_channel, 'Unknown') AS booking_channel,
    CASE
        WHEN status = 'Completed' THEN ticket_price
        ELSE 0
    END AS ticket_revenue
FROM park_entries_bronze
WHERE entry_id IS NOT NULL
  AND attraction_id IS NOT NULL
  AND ticket_price > 0
  AND wait_minutes >= 0;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) AS silver_records
FROM park_entries_silver;

silver_records
128


In [0]:
SELECT *
FROM park_entries_silver
WHERE booking_channel = 'Unknown';

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,ticket_revenue
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,Unknown,350.0
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,400.0,Completed,Unknown,400.0
1087,V537,A01,2026-10-11,10:00:00,75,4,Standard,350.0,Completed,Unknown,350.0


In [0]:
CREATE OR REPLACE TABLE park_gold_attraction
USING DELTA
AS
SELECT
    attraction_id,
    COUNT(*) AS total_entries,
    SUM(ticket_revenue) AS total_revenue,
    ROUND(AVG(wait_minutes), 2) AS average_wait
FROM park_entries_silver
GROUP BY attraction_id;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE park_gold_zone
USING DELTA
AS
SELECT
    attractions.location.zone AS zone,
    COUNT(*) AS total_entries,
    SUM(entries.ticket_revenue) AS total_revenue,
    ROUND(AVG(entries.wait_minutes), 2) AS average_wait
FROM park_entries_silver AS entries
INNER JOIN tp_attractions AS attractions
    ON entries.attraction_id = attractions.attraction_id
GROUP BY attractions.location.zone;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM park_gold_attraction
ORDER BY attraction_id;

attraction_id,total_entries,total_revenue,average_wait
A01,12,4800.0,37.5
A02,10,3500.0,29.0
A03,12,4100.0,23.33
A04,16,5800.0,30.0
A05,11,5850.0,20.45
A06,12,5500.0,27.5
A07,14,5100.0,33.93
A08,11,4200.0,35.0
A09,12,3200.0,34.17
A10,17,6400.0,32.35


In [0]:
SELECT *
FROM park_gold_zone
ORDER BY zone;

zone,total_entries,total_revenue,average_wait
East,39,15400.0,30.0
North,35,13850.0,31.0
South,26,9200.0,29.04
West,27,10000.0,32.04


In [0]:
SELECT
    attraction_id,
    SUM(ticket_revenue) AS total_revenue
FROM park_entries_silver
GROUP BY attraction_id
ORDER BY attraction_id;

attraction_id,total_revenue
A01,4800.0
A02,3500.0
A03,4100.0
A04,5800.0
A05,5850.0
A06,5500.0
A07,5100.0
A08,4200.0
A09,3200.0
A10,6400.0


Databricks visualization. Run in Databricks to view.

In [0]:
SELECT
    ticket_type,
    COUNT(*) AS ticket_count
FROM park_entries_silver
WHERE status = 'Completed'
GROUP BY ticket_type
ORDER BY ticket_count DESC;

ticket_type,ticket_count
Standard,57
Express,31
VIP,10


Databricks visualization. Run in Databricks to view.

In [0]:
SELECT
    visit_date,
    SUM(ticket_revenue) AS daily_revenue
FROM park_entries_silver
GROUP BY visit_date
ORDER BY visit_date;

visit_date,daily_revenue
2026-10-01,5250.0
2026-10-02,1000.0
2026-10-03,4500.0
2026-10-04,2850.0
2026-10-05,1550.0
2026-10-06,1950.0
2026-10-07,2250.0
2026-10-08,2900.0
2026-10-09,2350.0
2026-10-10,1750.0


Databricks visualization. Run in Databricks to view.

In [0]:
SELECT
    attraction_id,
    AVG(wait_minutes) AS average_wait,
    SUM(ticket_revenue) AS total_revenue
FROM park_entries_silver
GROUP BY attraction_id;

attraction_id,average_wait,total_revenue
A08,35.0,4200.0
A01,37.5,4800.0
A04,30.0,5800.0
A05,20.454545454545453,5850.0
A07,33.92857142857143,5100.0
A06,27.5,5500.0
A02,29.0,3500.0
A09,34.166666666666664,3200.0
A10,32.35294117647059,6400.0
A03,23.333333333333332,4100.0


Databricks visualization. Run in Databricks to view.

In [0]:
SELECT entry_id, ticket_price, status
FROM park_entries_silver
ORDER BY entry_id
LIMIT 5;

entry_id,ticket_price,status
1001,350.0,Cancelled
1002,350.0,Completed
1003,350.0,Completed
1004,650.0,Completed
1005,350.0,Completed


In [0]:
CREATE OR REPLACE TABLE park_entries_work
USING DELTA
AS
SELECT *
FROM park_entries_silver;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE park_entries_updates
USING DELTA
AS
SELECT
    1002 AS entry_id,
    400 AS ticket_price,
    'Completed' AS status;

num_affected_rows,num_inserted_rows


In [0]:
MERGE INTO park_entries_work AS target
USING park_entries_updates AS source
ON target.entry_id = source.entry_id

WHEN MATCHED THEN
UPDATE SET
    target.ticket_price = source.ticket_price,
    target.status = source.status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
SELECT entry_id, ticket_price, status
FROM park_entries_work
WHERE entry_id = 1002;

entry_id,ticket_price,status
1002,400.0,Completed


In [0]:
SELECT entry_id, ticket_price, status
FROM park_entries_silver
WHERE entry_id = 1002;

entry_id,ticket_price,status
1002,350.0,Completed


In [0]:
DESCRIBE HISTORY park_entries_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-11T11:19:33.000Z,74600853328676,luxya2026@gmail.com,MERGE,"Map(predicate -> [""(entry_id#29173 = entry_id#29188)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(2691849200926474),e66a23d9-ab11-4c01-81c4-e4a0823c22d3,1111-090100-bzyoeck0-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3327, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 2531, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 906, numTargetRowsUpdated -> 1, numOutputRows -> 1, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1590)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T11:18:58.000Z,74600853328676,luxya2026@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2691849200926474),3983f78d-3bb6-4da5-9402-7db192a8f61b,1111-090100-bzyoeck0-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 128, numOutputBytes -> 5061)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
SELECT entry_id, ticket_price, status
FROM park_entries_work VERSION AS OF 0
WHERE entry_id = 1002;

entry_id,ticket_price,status
1002,350.0,Completed


In [0]:
SELECT entry_id, ticket_price, status
FROM park_entries_work VERSION AS OF 1
WHERE entry_id = 1002;

entry_id,ticket_price,status
1002,400.0,Completed
